In [1]:
!pip install aiogram python-dotenv llama-index llama-index-vector-stores-chroma llama-index-embeddings-huggingface llama-index-llms-ollama chromadb sentence-transformers nest-asyncio

In [2]:
import os
import asyncio
import nest_asyncio
from dotenv import load_dotenv

nest_asyncio.apply()

In [3]:
load_dotenv()
BOT_TOKEN = os.getenv("TELEGRAM_BOT_API_KEY")
if BOT_TOKEN:
    print("Token is loaded!!!")
    print(BOT_TOKEN)
else: 
    print("Token not found!!!")

Token is loaded!!!
8774285768:AAF27o0Z2lI_4y_9DCzlw5nfRGnX-vIcFaY


In [4]:
import chromadb
from llama_index.core import StorageContext, VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.llms.ollama import Ollama
from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core.prompts import PromptTemplate
from llama_index.core.schema import Document

Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
Settings.llm = Ollama(model="llama3.2:3b", base_url="http://localhost:11434", request_timeout=120.0)

chroma_client = chromadb.HttpClient(host="localhost", port=8000)
try:
    chroma_client.delete_collection("rag_tg_bot_collection")
except:
    pass

chroma_collection = chroma_client.get_or_create_collection("rag_tg_bot_collection")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

print("RAG infrastructure is set up!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RAG infrastructure is set up!


In [5]:
!pip install python-docx

In [6]:
from docx import Document as DocxDocument
from llama_index.core.schema import TextNode
import uuid

def read_docx(file_path):
    doc = DocxDocument(file_path)
    return [p.text.strip() for p in doc.paragraphs if p.text.strip()]


def hybrid_chunking(file_path, chunk_size=256, chunk_overlap=20, para_threshold=256):
    paragraphs = read_docx(file_path)

    splitter = TokenTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    
    chunks=[]
    small_paragraphs = []
    
    for para in paragraphs:
        if len(splitter.split_text(para)) > 1:
            if small_paragraphs:
                chunks.append(" ".join(small_paragraphs))
                small_paragraphs = []

            sub_chunks = splitter.split_text(para)
            chunks.extend(sub_chunks)
        else:
            small_paragraphs.append(para)

            if len(" ".join(small_paragraphs)) > chunk_size:
                combined = " ".join(small_paragraphs)
                chunks.extend(splitter.split_text(combined)) 
                small_paragraphs = []

    if small_paragraphs:
        combined = " ".join(small_paragraphs)
        chunks.extend(splitter.split_text(combined)) 

    nodes = []
    
    for chunk in chunks:
        node = TextNode(
            text=chunk,
            metadata={
                "source": file_path,
                "chunk_id": uuid.uuid4().hex[:8]
            }
        )
        nodes.append(node)

    return nodes


nodes = hybrid_chunking(file_path='/Users/michael/ml-course-inno/report.docx', chunk_size=256, chunk_overlap=20)


try:
    chroma_client.delete_collection("rag_tg_bot_collection")
except:
    pass

chroma_collection = chroma_client.get_or_create_collection("rag_tg_bot_collection")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex(nodes=nodes, storage_context=storage_context, show_progress=True)
print(f"Chunks loaded to Chroma: {chroma_collection.count()}")

Generating embeddings:   0%|          | 0/49 [00:00<?, ?it/s]

Chunks loaded to Chroma: 49


In [7]:
qa_prompt_tmpl_str = (
    "Контекстная информация приведена ниже:\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Учитывая информацию из контекста и не используя ранее полученные знания, "
    "ответь на вопрос пользователя.\n"
    "Правила ответа:\n"
    "1. Отвечай СТРОГО на основе предоставленного выше контекста.\n"
    "2. Если в контексте прямо НЕТ ответа на вопрос, ответь строго фразой: "
    "'К сожалению, в предоставленных инструкциях нет информации об этом.'\n"
    "3. Будь лаконичен, отвечай четко и по делу.\n\n"
    "Вопрос пользователя: {query_str}\n"
    "Ответ:"
)

qa_prompt_template = PromptTemplate(qa_prompt_tmpl_str)

query_engine = index.as_query_engine(similarity_top_k=2, text_qa_template=qa_prompt_template)

print("RAG Query Engine is set up and ready!")

RAG Query Engine is set up and ready!


In [8]:
import sys
!{sys.executable} -m pip install aiogram

In [9]:
from aiogram import Bot, Dispatcher, types, html
from aiogram.filters import CommandStart
from aiogram.enums import ChatAction, ParseMode

bot = Bot(token=BOT_TOKEN)
dp = Dispatcher()

@dp.message(CommandStart())
async def start_handler(message: types.Message):
    welcome_text = (
        f"Привет, {html.bold(message.from_user.first_name)}! \n\n"
        "Я RAG-ассистент по очету по практике.\n"
        "Задай мне любой вопрос по отчету и я отвечу!"
    )
    await message.answer(welcome_text, parse_mode=ParseMode.HTML)

@dp.message()
async def rag_message_handler(message: types.Message):
    await bot.send_chat_action(chat_id=message.chat.id, action=ChatAction.TYPING)
    
    user_query = message.text
    
    try:
        response = await query_engine.aquery(user_query)
        
        answer_text = f"{response.response}\n\n"
        
        if hasattr(response, "source_nodes") and response.source_nodes:
            answer_text += f" {html.bold('Использованные источники из ChromaDB:')}\n"
            for idx, node_with_score in enumerate(response.source_nodes):
                chunk_id = node_with_score.node.metadata.get('chunk_id', 'N/A')
                score = node_with_score.score
                answer_text += f"Chunk #{chunk_id} (Релевантность: {score:.2f})\n"
        
        await message.answer(answer_text, parse_mode=ParseMode.HTML)
        
    except Exception as e:
        await message.answer(f"Произошла ошибка при обработке запроса: {e}")


print("Bot is set up and ready!")

Bot is set up and ready!


In [ ]:
async def main():
    print("Bot is running and ready to work!!!")
    await bot.delete_webhook(drop_pending_updates=True)
    await dp.start_polling(bot)

if __name__ == "__main__":
    asyncio.run(main())

Bot is running and ready to work!!!
